In [15]:
# =============================================================================
# CELL 1: CONFIGURATION
# =============================================================================

granularity = 'm'  # 'q' = quarterly, 'm' = monthly, 'w' = weekly

START_DATE = '2026-03-01'
END_DATE = None

run_every_query = False  # True = run SQL; False = use cache/ pickles from ste_ragu_postintegration

DATE_COL_MAP = {'q': 'book_date', 'm': 'book_date', 'w': 'app_date'}

LOBS = ['STE']

BASELINES = {
    'STE': {'ltv': 1.94, 'new_recovery_unadjusted': 0.557, 'apr': 0.25},
}

MODEL_PARAMS = {
    'mean_unit_loss': 0.5,
    'unit_loss_to_model_score': 0.02,
}

PRICING_SCALAR = 1.0  # STE does NOT use DLA

DIAG_N_VINTAGES = 6

EXCEL_OUTPUT = 'STE_gl_diagnostics.xlsx'

MODEL_SCORE_THRESHOLD = 140

CATEGORIES = {
    'STE': {
        'ula_filter': None,
        'weekly_filter': None,
    },
    'Downcredit': {
        'ula_filter': lambda df: df[df['cd_model_score'] < MODEL_SCORE_THRESHOLD],
        'weekly_filter': lambda df: df[df['con_risk_model_score'] < MODEL_SCORE_THRESHOLD],
    },
    'Upcredit': {
        'ula_filter': lambda df: df[df['cd_model_score'] >= MODEL_SCORE_THRESHOLD],
        'weekly_filter': lambda df: df[df['con_risk_model_score'] >= MODEL_SCORE_THRESHOLD],
    },
}

In [16]:
# =============================================================================
# CELL 2: IMPORTS AND DERIVED CONFIGURATION
# =============================================================================
import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import math
import os
import openpyxl

pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)

date_col = DATE_COL_MAP[granularity]
start_date = pd.Timestamp(START_DATE)
end_date = pd.Timestamp(END_DATE) if END_DATE else pd.Timestamp.today().normalize()

PERIOD_FREQ_MAP = {'q': 'Q', 'm': 'M', 'w': 'W-SAT'}
period_freq = PERIOD_FREQ_MAP[granularity]

start_period = start_date.to_period(period_freq)
end_period = end_date.to_period(period_freq)

min_date_sql = f"'{START_DATE}'"

print(f"Granularity: {granularity}")
print(f"Date column: {date_col}")
print(f"Period range: {start_period} to {end_period}")

Granularity: m
Date column: book_date
Period range: 2026-03 to 2026-08


In [17]:
# =============================================================================
# CELL 3: UTILITY FUNCTIONS
# =============================================================================

def run_sql(filename, sub_list=None, connection=None, filename_is_query=False):
    if sub_list is None:
        sub_list = []
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    for text, var in sub_list:
        query = query.replace(text, var)
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    else:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings("default", category=UserWarning)
        return df


def store_pickle(data, filename):
    if isinstance(data, str):
        data, filename = filename, data
    with open(filename, 'wb') as file:
        pickle.dump(data, file)


def get_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)


def cached_sql(filename, pickle_name, sub_list=None, connection=None, force_refresh=False):
    if force_refresh or not os.path.exists(pickle_name):
        df = run_sql(filename, sub_list=sub_list, connection=connection)
        store_pickle(df, pickle_name)
        return df
    return get_pickle(pickle_name)


def weighted_average_and_sum(group, metrics):
    if isinstance(metrics, str):
        weighted_avg = (group[metrics] * group.amt_financed).sum() / group.amt_financed.sum()
        return pd.Series({metrics: weighted_avg, 'amt_financed': group.amt_financed.sum()})
    result_dict = {'amt_financed': group.amt_financed.sum()}
    for metric in metrics:
        weighted_avg = (group[metric] * group.amt_financed).sum() / group.amt_financed.sum()
        result_dict[metric] = weighted_avg
    return pd.Series(result_dict)


def format_vintage(period_series):
    if len(period_series) == 0:
        return period_series.astype(str)
    freq = period_series.iloc[0].freqstr
    if freq == 'Q-DEC':
        return period_series.dt.year.astype(str) + ' Q' + period_series.dt.quarter.astype(str)
    elif freq == 'M':
        return period_series.dt.year.astype(str) + ' M' + period_series.dt.month.astype(str).str.zfill(2)
    return period_series.astype(str)

In [18]:
# =============================================================================
# CELL 4: ULA MULTIPLIER FUNCTIONS (NONKMX)
# =============================================================================

def get_ula_multiplier_nonkmx(ula_df, leave_out='None'):
    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Previous ACA chargeoff':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.prev_co_flag
    if leave_out != 'Small amount financed':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.small_amt_financed_flag * (1 - ula_df.pricing_change_flag)
    if leave_out != 'Zero cash down':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.zero_cash_down_flag * (1 - ula_df.pricing_change_flag)
    if leave_out != 'High mileage vehicle':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_mileage_vehicle_flag
    if leave_out != 'High PTI':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_pti_flag * (1 - ula_df.pricing_change_flag)
    if leave_out != 'Car make':
        ula_df.loss_multiplier *= (1 + 0.1 * ula_df.car_make_penalty_flag
                                   - 0.1 * ula_df.car_make_benefit_flag
                                   - 0.1 * ula_df.pricing_change_flag * ula_df.car_make_benefit_flag)
    if leave_out != 'Theft risk':
        ula_df.loss_multiplier *= (0.987 + 0.099 * ula_df.theft_risk_flag * (1 - ula_df.pricing_change_flag)
                                   + 0.013 * ula_df.pricing_change_flag)
    if leave_out != 'MCY high model score, low mileage':
        ula_df.loss_multiplier *= 1 - 0.2 * ula_df.mcy_low_mileage_flag
    if leave_out != 'Weekday/weekend decision':
        ula_df.loss_multiplier *= 1 - 0.05 * ula_df.weekend_flag + 0.02 * ula_df.weekday_flag
    if leave_out != 'Student Loans':
        ula_df.loss_multiplier *= 1 + (0.1 * ula_df.student_loan_flag
            - np.minimum(np.maximum((ula_df.cd_model_score - (130 - 2 * ula_df.ent_flag)) * 0.006, 0), 0.03)
            * (1 - ula_df.student_loan_flag)) * ula_df.student_loans_cutoff_date
    if leave_out != 'Low PTI':
        ula_df.loss_multiplier *= 1 + (-0.03 * np.minimum(ula_df.cd_model_score, 135) + 3.9) * ula_df.low_pti_flag
    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loss_multiplier *= 0.966 + 0.273 * ula_df.nonkmx_chime_flag
    if leave_out != 'Employment type':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.seasonal_employment_flag - 0.1 * ula_df.waiter_employment_flag
    if leave_out != 'Authorized tradelines':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.nonkmx_auth_tradelines_flag
    if leave_out != 'Fraud':
        ula_df.loss_multiplier *= 1 + (ula_df.fraud_adjustment - 1)
    if leave_out != 'Driver flag':
        ula_df.loss_multiplier *= 1 + 0.15 * ula_df.driver_flag
    if leave_out != 'Clip':
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.8, 2)
    if leave_out != 'Dealer Level (Non-KMX)':
        ula_df.loss_multiplier *= ula_df.pricing_scalar
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.7, 1.4)

    return ula_df


def get_ula_multiplier_nonkmx_diag(ula_df, leave_out='None', verbose=True):
    """Identical logic to get_ula_multiplier_nonkmx, plus per-step mean tracking."""
    steps = {}
    def _record(label, df):
        steps[label] = df.loss_multiplier.mean()

    ula_df['loss_multiplier'] = 1.0
    _record('00_initial', ula_df)
    if verbose:
        print(f"00_initial  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Previous ACA chargeoff':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.prev_co_flag
    _record('01_prev_aca_chargeoff', ula_df)
    if verbose:
        print(f"01_prev_co  | flag mean: {ula_df.prev_co_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Small amount financed':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.small_amt_financed_flag * (1 - ula_df.pricing_change_flag)
    _record('02_small_amt_financed', ula_df)
    if verbose:
        print(f"02_small_af | flag mean: {ula_df.small_amt_financed_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Zero cash down':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.zero_cash_down_flag * (1 - ula_df.pricing_change_flag)
    _record('03_zero_cash_down', ula_df)
    if verbose:
        print(f"03_zero_cd  | flag mean: {ula_df.zero_cash_down_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'High mileage vehicle':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_mileage_vehicle_flag
    _record('04_high_mileage', ula_df)
    if verbose:
        print(f"04_high_mi  | flag mean: {ula_df.high_mileage_vehicle_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'High PTI':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_pti_flag * (1 - ula_df.pricing_change_flag)
    _record('05_high_pti', ula_df)
    if verbose:
        print(f"05_high_pti | flag mean: {ula_df.high_pti_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Car make':
        ula_df.loss_multiplier *= (1 + 0.1 * ula_df.car_make_penalty_flag
                                   - 0.1 * ula_df.car_make_benefit_flag
                                   - 0.1 * ula_df.pricing_change_flag * ula_df.car_make_benefit_flag)
    _record('06_car_make', ula_df)
    if verbose:
        print(f"06_car_make | penalty: {ula_df.car_make_penalty_flag.mean():.6f}  benefit: {ula_df.car_make_benefit_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Theft risk':
        ula_df.loss_multiplier *= (0.987 + 0.099 * ula_df.theft_risk_flag * (1 - ula_df.pricing_change_flag)
                                   + 0.013 * ula_df.pricing_change_flag)
    _record('07_theft_risk', ula_df)
    if verbose:
        print(f"07_theft    | flag mean: {ula_df.theft_risk_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'MCY high model score, low mileage':
        ula_df.loss_multiplier *= 1 - 0.2 * ula_df.mcy_low_mileage_flag
    _record('08_mcy_low_mileage', ula_df)
    if verbose:
        print(f"08_mcy_low  | flag mean: {ula_df.mcy_low_mileage_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Weekday/weekend decision':
        ula_df.loss_multiplier *= 1 - 0.05 * ula_df.weekend_flag + 0.02 * ula_df.weekday_flag
    _record('09_weekend_weekday', ula_df)
    if verbose:
        print(f"09_wknd/day | weekend: {ula_df.weekend_flag.mean():.6f}  weekday: {ula_df.weekday_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Student Loans':
        ula_df.loss_multiplier *= 1 + (0.1 * ula_df.student_loan_flag
            - np.minimum(np.maximum((ula_df.cd_model_score - (130 - 2 * ula_df.ent_flag)) * 0.006, 0), 0.03)
            * (1 - ula_df.student_loan_flag)) * (ula_df.student_loans_cutoff_date)
    _record('10_student_loans', ula_df)
    if verbose:
        print(f"10_stud_ln  | flag mean: {ula_df.student_loan_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Low PTI':
        ula_df.loss_multiplier *= 1 + (-0.03 * np.minimum(ula_df.cd_model_score, 135) + 3.9) * ula_df.low_pti_flag
    _record('11_low_pti', ula_df)
    if verbose:
        print(f"11_low_pti  | flag mean: {ula_df.low_pti_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loss_multiplier *= 0.966 + 0.273 * ula_df.nonkmx_chime_flag
    _record('12_chime', ula_df)
    if verbose:
        print(f"12_chime    | flag mean: {ula_df.nonkmx_chime_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Employment type':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.seasonal_employment_flag - 0.1 * ula_df.waiter_employment_flag
    _record('13_employment_type', ula_df)
    if verbose:
        n = len(ula_df)
        print(f"13_employ   | seasonal: {ula_df.seasonal_employment_flag.sum()/n:.6f}  waiter: {ula_df.waiter_employment_flag.sum()/n:.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Authorized tradelines':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.nonkmx_auth_tradelines_flag
    _record('14_auth_tradelines', ula_df)
    if verbose:
        print(f"14_auth_tl  | flag mean: {ula_df.nonkmx_auth_tradelines_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Fraud':
        ula_df.loss_multiplier *= 1 + (ula_df.fraud_adjustment - 1)
    _record('15_fraud', ula_df)
    if verbose:
        print(f"15_fraud    | fraud_adj mean: {ula_df.fraud_adjustment.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Driver flag':
        ula_df.loss_multiplier *= 1 + 0.15 * ula_df.driver_flag
    _record('16_driver_flag', ula_df)
    if verbose:
        print(f"16_driver   | flag mean: {ula_df.driver_flag.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Clip':
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.8, 2)

    if leave_out != 'Dealer Level (Non-KMX)':
        ula_df.loss_multiplier *= ula_df.pricing_scalar
    _record('17_pricing_scalar', ula_df)
    if verbose:
        print(f"17_pricing  | scalar mean: {ula_df.pricing_scalar.mean():.6f}  | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}")

    if leave_out != 'Dealer Level (Non-KMX)':
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.7, 1.4)
    _record('18_final', ula_df)
    if verbose:
        print(f"18_final    | loss_multiplier mean: {ula_df.loss_multiplier.mean():.6f}  (post all clips)")

    n = len(ula_df)
    flags = {
        'prev_co_flag': ula_df.prev_co_flag.mean(),
        'small_amt_financed_flag': ula_df.small_amt_financed_flag.mean(),
        'pricing_change_flag': ula_df.pricing_change_flag.mean(),
        'zero_cash_down_flag': ula_df.zero_cash_down_flag.mean(),
        'high_mileage_vehicle_flag': ula_df.high_mileage_vehicle_flag.mean(),
        'high_pti_flag': ula_df.high_pti_flag.mean(),
        'car_make_penalty_flag': ula_df.car_make_penalty_flag.mean(),
        'car_make_benefit_flag': ula_df.car_make_benefit_flag.mean(),
        'theft_risk_flag': ula_df.theft_risk_flag.mean(),
        'mcy_low_mileage_flag': ula_df.mcy_low_mileage_flag.mean(),
        'weekend_flag': ula_df.weekend_flag.mean(),
        'weekday_flag': ula_df.weekday_flag.mean(),
        'student_loan_flag': ula_df.student_loan_flag.mean(),
        'student_loans_cutoff_date': ula_df.student_loans_cutoff_date.mean(),
        'low_pti_flag': ula_df.low_pti_flag.mean(),
        'nonkmx_chime_flag': ula_df.nonkmx_chime_flag.mean(),
        'seasonal_employment_pct': ula_df.seasonal_employment_flag.sum() / n,
        'waiter_employment_pct': ula_df.waiter_employment_flag.sum() / n,
        'nonkmx_auth_tradelines_flag': ula_df.nonkmx_auth_tradelines_flag.mean(),
        'fraud_adjustment_mean': ula_df.fraud_adjustment.mean(),
        'driver_flag': ula_df.driver_flag.mean(),
        'pricing_scalar_mean': ula_df.pricing_scalar.mean(),
    }

    return ula_df, pd.Series(steps), pd.Series(flags)

In [19]:
# =============================================================================
# CELL 5: DATA FETCH (shared cache/ from ste_ragu_postintegration)
# =============================================================================
# Pickles already contain temp-table-derived data (employment, credit attrs,
# blackbook, fraud) baked in from the JOINs. Temp table execution is only
# needed when run_every_query = True.

os.makedirs('cache', exist_ok=True)
force = run_every_query

need_conn = force or not all(
    os.path.exists(p) for p in ('cache/ste_ula_v1.pkl', 'cache/ste_recovery_v1.pkl', 'cache/ste_weekly_v1.pkl')
)

if need_conn:
    conn = pyodbc.connect("DSN=Redshift_prod_new")

    with open('ste_ragu_temptables.txt', 'r') as f:
        conn.execute(f.read().strip())
    print('Temp tables created')

    ula_df_total = cached_sql('ste_ragu_ula.txt', 'cache/ste_ula_v1.pkl',
                              connection=conn, force_refresh=force)
    print(f'ULA ready: {len(ula_df_total):,} records')

    new_recovery = cached_sql('ste_ragu_recovery.txt', 'cache/ste_recovery_v1.pkl',
                              connection=conn, force_refresh=force)
    print(f'Recovery ready: {len(new_recovery):,} records')

    ste_weekly_raw = cached_sql('ste_ragu_weekly.txt', 'cache/ste_weekly_v1.pkl',
                                connection=conn, force_refresh=force)
    print(f'STE weekly metrics ready: {len(ste_weekly_raw):,} records')

    conn.close()
else:
    ula_df_total = get_pickle('cache/ste_ula_v1.pkl')
    new_recovery = get_pickle('cache/ste_recovery_v1.pkl')
    ste_weekly_raw = get_pickle('cache/ste_weekly_v1.pkl')
    print('ULA, Recovery, STE weekly loaded from cache')

print(f"ULA records: {len(ula_df_total):,}")
print("[PROGRESS] Data Fetch Complete")


ULA, Recovery, STE weekly loaded from cache
ULA records: 81,998
[PROGRESS] Data Fetch Complete


In [20]:
# =============================================================================
# CELL 6: PERIOD ASSIGNMENT, FILTERING, FLAG CONSTRUCTION
# (mirrors ste_ragu_postintegration Cells 6+7 exactly)
# =============================================================================

ula_df_total = ula_df_total[ula_df_total.lob != 'Core']

ula_df_total['lob'] = 'STE'

ula_df_total['app_date'] = pd.to_datetime(ula_df_total['app_date'])
ula_df_total['book_date'] = pd.to_datetime(ula_df_total['book_date'])
new_recovery['app_date'] = pd.to_datetime(new_recovery['app_date'])
new_recovery['book_date'] = pd.to_datetime(new_recovery['book_date'])

for df in [ula_df_total, new_recovery]:
    df['quarter'] = pd.to_datetime(df[date_col]).dt.to_period('Q')
    df['month'] = pd.to_datetime(df[date_col]).dt.to_period('M')
    df['week'] = pd.to_datetime(df[date_col]).dt.to_period('W-SAT')
    period_key = {'q': 'quarter', 'm': 'month', 'w': 'week'}
    df['period'] = df[period_key[granularity]]

for df in [ula_df_total, new_recovery]:
    mask = (df['period'] >= start_period) & (df['period'] <= end_period)
    df.drop(df[~mask].index, inplace=True)

ula_df_total['book_week'] = ula_df_total['book_week'].astype(str)

date_col_str = f'{date_col}_str'
ula_df_total[date_col_str] = ula_df_total[date_col].astype(str)

# --- STE caps (BEFORE processing, matching ste_ragu_postintegration Cell 6) ---
# ula_df_total = ula_df_total[ula_df_total.amt_financed <= 75000]
ula_df_total['bbltv'] = ula_df_total.amt_financed / ula_df_total.bbvalue.replace(0, np.nan)
ula_df_total = ula_df_total[
    (ula_df_total.bbltv <= 10.0) |
    (ula_df_total.bbvalue.isna()) |
    (ula_df_total.bbvalue == 0)
]
ula_df_total = ula_df_total[ula_df_total.pti <= 0.6]
ula_df_total = ula_df_total[ula_df_total.total_income <= 200000]

print(f"Periods in data: {ula_df_total['period'].nunique()}")
print(f"Period range: {ula_df_total['period'].min()} to {ula_df_total['period'].max()}")
print(f"ULA after caps: {len(ula_df_total):,}")

# --- ULA Processing (matching ste_ragu_postintegration Cell 7) ---
ula_df_total['mtn_3_1_flag'] = ula_df_total.mtn_model == 'MTN3.1'
ula_df_total['vehicle_age'] = np.maximum(ula_df_total[date_col_str].str[:4].astype(int) - ula_df_total.model_year, 1/365)
ula_df_total.tradein_value = ula_df_total.tradein_value.fillna(0)
ula_df_total.make = ula_df_total.make.str.upper().str[:3]
ula_df_total.lob_or_bucket = np.select(
    [ula_df_total.lob_or_bucket.isna() & ula_df_total.lob.isin(['Core', 'FRN']),
     ula_df_total.lob_or_bucket.isna() & ~ula_df_total.lob.isin(['Core', 'FRN'])],
    ['D', 'C'], default=ula_df_total.lob_or_bucket)
ula_df_total['continuous_vehicle_age'] = (ula_df_total[date_col_str].str[:4].astype(int)
    + ula_df_total[date_col_str].str[5:7].astype(int) / 12
    - (ula_df_total.model_year - 0.25) - 1)

ula_df_total['pricing_scalar'] = PRICING_SCALAR

# --- Driver Flag ---
warnings.filterwarnings('ignore', category=UserWarning)
ula_df_total.job_company = ula_df_total.job_company.fillna('')
ula_df_total['driver_flag'] = np.where(
    ula_df_total.job_company.str.contains('(LYFT)|(UBER)|(GRUB ?HUB)|(DOOR ?DASH)|(GO ?PUFF)|(POST ?MATE)|(INSTA ?CART)|(DOMINO)|(PAPA J)|(PIZZA)|(JIMMY ?JOHN)|(SELF)'),
    1, 0)
warnings.filterwarnings('default', category=UserWarning)

# --- NA Handling ---
ula_df_total = ula_df_total.dropna(subset=['lob'])
ula_df_total.cash_down = ula_df_total.cash_down.fillna(0)
ula_df_total.specialty_dealer = ula_df_total.specialty_dealer.fillna('not specialty')
ula_df_total.prev_co_count = ula_df_total.prev_co_count.fillna(0)

# --- NonKMX Flags (STE-specific: PTI > 0.2, no MCY/ENT guards) ---
ula_df_total['ent_fld_flag'] = False
ula_df_total['small_amt_financed_flag'] = (ula_df_total.bbvalue > 0) & (ula_df_total.bbvalue < 5000) & (ula_df_total.amt_financed < 4500)
ula_df_total['zero_cash_down_flag'] = (ula_df_total.cash_down <= 250) & (ula_df_total.tradein_value < 3000)
ula_df_total['high_mileage_vehicle_flag'] = (ula_df_total.mileage >= 100000) & (ula_df_total.cd_model_score < 130)
ula_df_total['high_pti_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_penalty_flag'] = ula_df_total.make.isin({'CAD', 'CHR', 'BMW', 'BUI', 'SUB'}) & (ula_df_total.cd_model_score < 130) & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_benefit_flag'] = (ula_df_total.cd_model_score >= 133) & ula_df_total.make.isin({'HON', 'TOY', 'LEX'}) & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['theft_risk_flag'] = ula_df_total.make.isin({'KIA', 'HYU'}) & (ula_df_total.specialty_dealer != 'Ally') & (ula_df_total.model_year >= 2015) & (ula_df_total.model_year <= 2021) & (ula_df_total[date_col_str] >= '2022-07-01') & (ula_df_total[date_col_str] < '2025-01-01')
ula_df_total['mcy_low_mileage_flag'] = False
ula_df_total['weekend_flag'] = ula_df_total.day_of_week.isin([0, 6])
ula_df_total['weekday_flag'] = ula_df_total.day_of_week.isin(range(1, 6))
ula_df_total['secured_credit_flag'] = ula_df_total.secured_credit_card
ula_df_total['chime_flag'] = ula_df_total.chime_indicator
ula_df_total['nonkmx_chime_flag'] = ula_df_total.nonkmx_chime_indicator
ula_df_total['seasonal_employment_flag'] = ula_df_total['seasonal_employment_flag'] == 1 if 'seasonal_employment_flag' in ula_df_total.columns else (ula_df_total.get('employment', '') == 'seasonal')
ula_df_total['waiter_employment_flag'] = ula_df_total['waiter_employment_flag'] == 1 if 'waiter_employment_flag' in ula_df_total.columns else (ula_df_total.get('employment', '') == 'waiter')
ula_df_total['nonkmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines > 0.2
ula_df_total['prev_co_flag'] = ula_df_total.prev_co_count > 0
ula_df_total['null_fico_w_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & (ula_df_total.vantage_score >= 300) & (ula_df_total.vantage_score <= 850)
ula_df_total['pricing_change_flag'] = ula_df_total[date_col_str] >= '2024-10-01'
ula_df_total['ent_flag'] = False
ula_df_total['student_loans_cutoff_date'] = ula_df_total[date_col_str] >= '2023-05-01'
ula_df_total['low_pti_flag'] = (ula_df_total.pti <= 0.05) & (ula_df_total.cd_model_score >= 130) & (ula_df_total.cb_flag)
ula_df_total['student_loan_flag'] = 0

# --- KMX Flags (retained for structural parity with ste_ragu_postintegration) ---
ula_df_total['high_sales_price_flag'] = ula_df_total.sale_price > 30000
ula_df_total['kmx_npc_flag'] = False
ula_df_total['high_pti_npc'] = ula_df_total.pti > 0.2
ula_df_total['job_time_flag'] = ula_df_total.employed_months < 6
ula_df_total['low_fico_flag'] = (ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 450)
ula_df_total['high_model_score_flag'] = ula_df_total.cd_model_score >= 146
ula_df_total['low_vantage_flag'] = (ula_df_total.fico_score < 300) & (ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450)
ula_df_total['louisiana_flag'] = ula_df_total.state == 'LA'
ula_df_total['txca_flag'] = ula_df_total.state.isin(['TX', 'CA'])
ula_df_total['georgia_flag'] = ula_df_total.state == 'GA'
ula_df_total['normal_pti_flag'] = ula_df_total.pti <= 0.2
ula_df_total['high_pti_tier_1_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.pti <= 0.25)
ula_df_total['high_pti_tier_2_flag'] = (ula_df_total.pti > 0.25) & (ula_df_total.pti <= 0.35)
ula_df_total['high_pti_tier_3_flag'] = ula_df_total.pti > 0.35
ula_df_total['existing_dq_flag'] = ula_df_total.existing_dq_count > 0
ula_df_total['kmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines >= 0.14
ula_df_total['low_bureau_flag'] = ((ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 475)) | ((ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450))
ula_df_total['soft_pull_flag'] = ula_df_total.pull_type.isin(['softpull', 'prequal'])
ula_df_total['cd_perc_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1)
ula_df_total['open_tl_flag'] = ula_df_total.open_tl == 0
ula_df_total['narrowed_soft_pull_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1) & ula_df_total.soft_pull_flag & (~ula_df_total.job_time_flag)

# --- Deduplicate driver flags ---
combined_driver_flag_df = ula_df_total.groupby('account_number').driver_flag.max().reset_index()
ula_df_total = ula_df_total.drop(columns='driver_flag').merge(combined_driver_flag_df, on='account_number')
ula_df_total = ula_df_total.drop(columns='job_company').drop_duplicates()

# --- Vintage strings ---
ula_df_total['vintage'] = format_vintage(ula_df_total['period'])
new_recovery['vintage'] = format_vintage(new_recovery['period'])

# --- Aggregate model scores from ULA data ---
ms_df = ula_df_total.groupby(['period', 'lob']).apply(
    weighted_average_and_sum, 'cd_model_score', include_groups=False
).reset_index()
ms_df = ms_df.rename(columns={'cd_model_score': 'model_score'})
ms_df['period'] = format_vintage(ms_df['period'])

print(f"ULA after refinement: {len(ula_df_total):,}")
print(f"Model scores aggregated: {len(ms_df)} period-LOB combinations")

# --- DUAL-PATH: Build ste_metrics_df from ste_ragu_weekly.txt ---
ste_filtered = ste_weekly_raw.copy()
ste_filtered['book_date'] = pd.to_datetime(ste_filtered['book_date'])
ste_filtered['period'] = pd.to_datetime(ste_filtered['book_date']).dt.to_period(period_freq)
ste_filtered = ste_filtered[(ste_filtered.period >= start_period) & (ste_filtered.period <= end_period)]
ste_filtered['vintage'] = format_vintage(ste_filtered['period'])

# ste_filtered = ste_filtered[ste_filtered['con_amount_financed_back'] <= 75000]
ste_filtered = ste_filtered[ste_filtered['con_pti_back'] <= 0.6]
ste_filtered = ste_filtered[ste_filtered['total_income'] <= 200000]
ste_filtered['bbltv'] = ste_filtered['con_amount_financed_back'] / ste_filtered['bb_value'].replace(0, np.nan)
ste_filtered = ste_filtered[
    (ste_filtered['bbltv'] <= 10.0) |
    (ste_filtered['bb_value'].isna()) |
    (ste_filtered['bb_value'] == 0)
]

def _build_ste_metrics(g):
    w = g['con_amount_financed_back']
    ms_valid = g['con_risk_model_score'].notnull()
    ltv_valid = g['bbltv'].notnull()
    return pd.Series({
        'model_score_wtd': (g.loc[ms_valid, 'con_risk_model_score'] * w[ms_valid]).sum() / w[ms_valid].sum() if ms_valid.any() else np.nan,
        'ltv_wtd': (g.loc[ltv_valid, 'bbltv'] * w[ltv_valid]).sum() / w[ltv_valid].sum() if ltv_valid.any() else np.nan,
        'apr_wtd': (g['con_apr'] * w).sum() / w.sum(),
        'amt_financed_total': w.sum(),
        'n_accounts': len(g),
    })

ste_metrics_df = ste_filtered.groupby('vintage').apply(_build_ste_metrics).reset_index()
print(f"\nste_metrics_df built: {len(ste_metrics_df)} vintages")
print(ste_metrics_df.head())

Periods in data: 6
Period range: 2026-03 to 2026-08
ULA after caps: 34,046
ULA after refinement: 13,635
Model scores aggregated: 6 period-LOB combinations

ste_metrics_df built: 6 vintages
    vintage  model_score_wtd   ltv_wtd   apr_wtd  amt_financed_total  \
0  2026 M03       134.393443  1.461419  0.228505        1.078094e+08   
1  2026 M04       135.073919  1.446427  0.226109        9.053636e+07   
2  2026 M05       136.244091  1.408017  0.224527        7.272996e+07   
3  2026 M06       136.526976  1.423415  0.223104        8.233838e+07   
4  2026 M07       137.337138  1.411365  0.220733        5.940834e+07   

   n_accounts  
0      3771.0  
1      2881.0  
2      2100.0  
3      2299.0  
4      1607.0  


In [21]:
# =============================================================================
# CELL 7: RAGU SCORE COMPUTATION (with STE dual-path override)
# (mirrors ste_ragu_postintegration Cell 8 exactly)
# =============================================================================

mean_unit_loss = MODEL_PARAMS['mean_unit_loss']
unit_loss_to_model_score = MODEL_PARAMS['unit_loss_to_model_score']


def get_ragu_score(vintage, lob, ula_df_total, new_recovery, ms_df, baseline_config, leave_out='None', ste_metrics_df=None):
    baseline_ltv = baseline_config['ltv']
    new_baseline_recovery_unadjusted_pct = baseline_config['new_recovery_unadjusted']
    baseline_apr = baseline_config['apr']

    ula_df = ula_df_total[(ula_df_total.vintage == vintage) & (ula_df_total.lob == lob)].copy()
    if len(ula_df) == 0:
        return None

    ula_df = get_ula_multiplier_nonkmx(ula_df, leave_out)

    ula_df = ula_df[['account_number', date_col, 'bbvalue', 'sale_price', 'amt_financed', 'lob_or_bucket', 'lob', 'loss_multiplier', 'apr']]

    nr = new_recovery[['account_number', 'new_recovery_multiplier']].drop_duplicates(subset='account_number', keep='first')
    mix_df = ula_df.merge(nr.rename(columns={'new_recovery_multiplier': 'recovery_multiplier'}),
                          on='account_number', how='left').drop_duplicates(subset='account_number', keep='first')
    mix_df['ltv'] = mix_df.amt_financed / mix_df.bbvalue

    bb_populated_df = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0)]
    if len(bb_populated_df) == 0:
        return None

    full_pop_metrics = bb_populated_df.groupby('lob').apply(
        weighted_average_and_sum,
        ['loss_multiplier', 'ltv', 'bbvalue', 'apr'],
        include_groups=False
    )

    recovery_df = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0) & mix_df['recovery_multiplier'].notna()].copy()
    recovery_df['recovery_unadjusted_multiplier'] = recovery_df['recovery_multiplier']
    recovery_metrics = recovery_df.groupby('lob').apply(
        weighted_average_and_sum,
        ['recovery_unadjusted_multiplier'],
        include_groups=False
    )
    recovery_metrics = recovery_metrics.drop(columns='amt_financed')

    grouped_mix_df = full_pop_metrics.join(recovery_metrics)
    vintage_ms_df = ms_df[ms_df['period'] == vintage].copy()

    index_name = grouped_mix_df.index.name
    if isinstance(index_name, str) and index_name in grouped_mix_df.columns:
        grouped_mix_df = grouped_mix_df.reset_index(drop=True)
    else:
        grouped_mix_df = grouped_mix_df.reset_index()

    full_df = grouped_mix_df.merge(vintage_ms_df, on='lob')
    if len(full_df) == 0:
        return None

    full_df['est_unit_loss'] = mean_unit_loss
    full_df['unit_loss_score'] = full_df.model_score + (1 - full_df.loss_multiplier) * full_df.est_unit_loss / unit_loss_to_model_score
    full_df = full_df.set_index('lob')

    full_df['ms_original'] = full_df.model_score.copy()
    full_df['baselined_unadjusted_recovery'] = (full_df.recovery_unadjusted_multiplier / new_baseline_recovery_unadjusted_pct).copy()

    full_df['gross_loss_impact'] = full_df['unit_loss_score'] - full_df['ms_original']
    full_df['recovery_impact'] = (full_df['unit_loss_score'] * full_df['est_unit_loss']
        * full_df['recovery_unadjusted_multiplier'] * (full_df['baselined_unadjusted_recovery'] - 1))
    full_df['ltv_impact'] = ((baseline_ltv / full_df['ltv']) - 1) * 17
    full_df['apr_impact'] = (baseline_apr - full_df['apr']) / 0.01 * 0.7
    full_df['ragu_score'] = (
        (1 - full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier) * full_df.unit_loss_score
        + full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier
        * full_df.unit_loss_score * full_df.baselined_unadjusted_recovery
        + full_df['ltv_impact']
        + full_df['apr_impact'])

    # --- STE dual-path override (MANDATORY) ---
    if ste_metrics_df is not None:
        ste_row = ste_metrics_df[ste_metrics_df.vintage == vintage]
        if len(ste_row) > 0:
            ste_row = ste_row.iloc[0]
            full_df['ms_original'] = ste_row['model_score_wtd']
            full_df['ltv'] = ste_row['ltv_wtd']
            full_df['apr'] = ste_row['apr_wtd']
            full_df['ltv_impact'] = ((baseline_ltv / full_df['ltv']) - 1) * 17
            full_df['apr_impact'] = (baseline_apr - full_df['apr']) / 0.01 * 0.7
            full_df['ragu_score'] = (
                (1 - full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier) * full_df.unit_loss_score
                + full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier
                * full_df.unit_loss_score * full_df.baselined_unadjusted_recovery
                + full_df['ltv_impact']
                + full_df['apr_impact'])

    full_df['vintage'] = vintage
    return full_df


all_vintages = sorted(ula_df_total['vintage'].unique())
results_by_category = {}

for cat_name, cat_cfg in CATEGORIES.items():
    cat_ula = cat_cfg['ula_filter'](ula_df_total) if cat_cfg['ula_filter'] else ula_df_total.copy()
    cat_weekly = cat_cfg['weekly_filter'](ste_filtered) if cat_cfg['weekly_filter'] else ste_filtered.copy()

    cat_ms_df = cat_ula.groupby(['period', 'lob']).apply(
        weighted_average_and_sum, 'cd_model_score', include_groups=False
    ).reset_index()
    cat_ms_df = cat_ms_df.rename(columns={'cd_model_score': 'model_score'})
    cat_ms_df['period'] = format_vintage(cat_ms_df['period'])

    cat_ste_metrics = cat_weekly.groupby('vintage').apply(_build_ste_metrics).reset_index()

    cat_results = {}
    for lob in LOBS:
        baseline_config = BASELINES[lob]
        lob_results = []
        for vintage in all_vintages:
            try:
                result = get_ragu_score(
                    vintage, lob, cat_ula, new_recovery, cat_ms_df, baseline_config,
                    ste_metrics_df=cat_ste_metrics
                )
                if result is not None:
                    lob_results.append(result)
            except Exception as e:
                print(f"Error: {cat_name} / {vintage} {lob}: {e}")
        if lob_results:
            cat_results[lob] = pd.concat(lob_results)
            print(f"{cat_name} / {lob}: {len(lob_results)} vintages ({len(cat_ula):,} ULA, {len(cat_weekly):,} weekly)")
        else:
            cat_results[lob] = pd.DataFrame()
            print(f"{cat_name} / {lob}: no results")
    results_by_category[cat_name] = cat_results

print(f"\nAll categories processed.")
print("[PROGRESS] Scoring Complete")


STE / STE: 6 vintages (13,635 ULA, 13,631 weekly)
Downcredit / STE: 6 vintages (10,411 ULA, 10,408 weekly)
Upcredit / STE: 6 vintages (3,224 ULA, 3,223 weekly)

All categories processed.
[PROGRESS] Scoring Complete


In [22]:
# =============================================================================
# CELL 8: DIAGNOSTICS - STEP-LEVEL ATTRIBUTION PER LOB
# =============================================================================

def build_output_df(results, wtd_mults, record_counts, ragu_gli_dict=None):
    if not results:
        return None
    step_names = list(next(iter(results.values())).keys())
    col_data = {}
    for (lob, vintage), step_dict in results.items():
        col_name = f"{lob} | {vintage}"
        col_data[col_name] = [step_dict.get(s) for s in step_names]
    diag_df = pd.DataFrame(col_data, index=step_names)
    summary = {}
    for (lob, vintage) in results.keys():
        col = f"{lob} | {vintage}"
        final_mult_mean = diag_df[col].iloc[-1]
        wtd_mult = wtd_mults.get((lob, vintage), float('nan'))
        if ragu_gli_dict is not None:
            gross_loss_impact = ragu_gli_dict.get((lob, vintage), 25 * (1 - wtd_mult))
        else:
            gross_loss_impact = 25 * (1 - wtd_mult)
        summary[col] = {
            '--- FINAL_MULT (mean)': final_mult_mean,
            '--- WTD_MULT_RAGU': wtd_mult,
            '--- GROSS_LOSS_IMPACT': gross_loss_impact,
            '--- N_RECORDS': record_counts.get((lob, vintage), 0),
        }
    summary_df = pd.DataFrame(summary)
    return pd.concat([diag_df, summary_df])


def build_flags_df(flag_results, record_counts):
    if not flag_results:
        return None
    flag_names = list(next(iter(flag_results.values())).keys())
    flag_col_data = {}
    for (lob, vintage), flag_dict in flag_results.items():
        col_name = f"{lob} | {vintage}"
        flag_col_data[col_name] = [flag_dict.get(f) for f in flag_names]
    flags_df = pd.DataFrame(flag_col_data, index=flag_names)
    n_row = {}
    for (lob, vintage) in flag_results.keys():
        n_row[f"{lob} | {vintage}"] = record_counts.get((lob, vintage), 0)
    flags_df.loc['--- N_RECORDS'] = n_row
    return flags_df


STEP_LABEL_MAP = {
    '00_initial':              'Initial (1.0)',
    '01_prev_aca_chargeoff':   'Previous ACA Chargeoff',
    '02_small_amt_financed':   'Small Amount Financed',
    '03_zero_cash_down':       'Zero Cash Down',
    '04_high_mileage':         'High Mileage Vehicle',
    '05_high_pti':             'High PTI',
    '06_car_make':             'Car Make',
    '07_theft_risk':           'Theft Risk',
    '08_mcy_low_mileage':      'MCY Low Mileage',
    '09_weekend_weekday':      'Weekend / Weekday',
    '10_student_loans':        'Student Loans',
    '11_low_pti':              'Low PTI',
    '12_chime':                'Chime / Secured Credit',
    '13_employment_type':      'Employment Type',
    '14_auth_tradelines':      'Authorized Tradelines',
    '15_fraud':                'Fraud Adjustment',
    '16_driver_flag':          'Driver Flag',
    '17_pricing_scalar':       'Dealer Level (Pricing Scalar)',
    '18_final':                'Final Clip',
}


def build_attribution_df(results, ragu_gli_dict):
    """
    Decompose gross_loss_impact across multiplier steps using logarithmic attribution.
    gross_loss_impact is sourced from get_ragu_score output to ensure the attribution
    decomposes the same value that appears in the RAGU score decomposition.
    """
    if not results:
        return None

    step_keys = list(next(iter(results.values())).keys())
    adjustment_keys = [k for k in step_keys if k != '00_initial']
    readable_labels = [STEP_LABEL_MAP.get(k, k) for k in adjustment_keys]

    col_data = {}
    for (lob, vintage), step_dict in results.items():
        col_name = f"{lob} | {vintage}"
        gross_loss_impact = ragu_gli_dict.get((lob, vintage), float('nan'))

        cumulative = [step_dict.get(k, float('nan')) for k in step_keys]
        ratios = []
        for i, k in enumerate(step_keys):
            if k == '00_initial':
                continue
            prev = cumulative[i - 1]
            curr = cumulative[i]
            if prev and prev != 0:
                ratios.append(curr / prev)
            else:
                ratios.append(1.0)

        final_mult = cumulative[-1]
        log_final = math.log(final_mult) if final_mult and final_mult > 0 and abs(final_mult - 1.0) > 1e-12 else None

        if log_final is None or pd.isna(gross_loss_impact):
            col_data[col_name] = [0.0] * len(adjustment_keys) + [gross_loss_impact if not pd.isna(gross_loss_impact) else 0.0]
        else:
            log_ratios = [math.log(r) if r and r > 0 else 0.0 for r in ratios]
            attributed = [(lr / log_final) * gross_loss_impact for lr in log_ratios]
            col_data[col_name] = attributed + [sum(attributed)]

    index_labels = readable_labels + ['--- TOTAL (check)']
    return pd.DataFrame(col_data, index=index_labels)


# --- Run diagnostics per category / LOB ---
diag_results_by_category = {}

for cat_name, cat_cfg in CATEGORIES.items():
    cat_ula = cat_cfg['ula_filter'](ula_df_total) if cat_cfg['ula_filter'] else ula_df_total.copy()
    cat_results_lob = results_by_category.get(cat_name, {})
    diag_results_by_category[cat_name] = {}

    for lob in LOBS:
        ula_lob = cat_ula[cat_ula.lob == lob].copy()
        if len(ula_lob) == 0:
            print(f'\n{cat_name} / {lob}: No data, skipping.')
            diag_results_by_category[cat_name][lob] = (None, None, None)
            continue

        ragu_gli_dict = {}
        model_df = cat_results_lob.get(lob)
        if model_df is not None and len(model_df) > 0:
            for _, row in model_df.reset_index().iterrows():
                ragu_gli_dict[(row['lob'], row['vintage'])] = row['gross_loss_impact']

        target_vintages = sorted(ula_lob.vintage.unique())[-DIAG_N_VINTAGES:]

        lob_results = {}
        lob_flag_results = {}
        lob_wtd_mults = {}
        lob_record_counts = {}

        print(f'\n{"="*60}')
        print(f'  DIAGNOSTICS: {cat_name} / {lob}')
        print(f'{"="*60}')

        for vintage in target_vintages:
            ula_vintage = ula_lob[ula_lob.vintage == vintage].copy()
            n = len(ula_vintage)
            if n == 0:
                continue

            print(f'\n{"="*60}')
            print(f'  {cat_name} / {lob} {vintage}  (n={n})')
            print(f'{"="*60}')

            ula_vintage_diag, steps, flags = get_ula_multiplier_nonkmx_diag(ula_vintage, leave_out='None', verbose=True)

            diag_mix = ula_vintage_diag[['account_number', 'bbvalue', 'amt_financed', 'loss_multiplier']].copy()
            nr = new_recovery[['account_number', 'new_recovery_multiplier']].drop_duplicates(
                subset='account_number', keep='first')
            diag_mix = diag_mix.merge(nr, on='account_number', how='left').drop_duplicates(
                subset='account_number', keep='first')
            bb_pop = diag_mix[diag_mix['bbvalue'].notna() & (diag_mix['bbvalue'] > 0)]
            if len(bb_pop) > 0 and bb_pop.amt_financed.sum() > 0:
                wtd_mult = (bb_pop.loss_multiplier * bb_pop.amt_financed).sum() / bb_pop.amt_financed.sum()
            else:
                wtd_mult = float('nan')

            ragu_gli = ragu_gli_dict.get((lob, vintage), float('nan'))
            print(f"  bb_populated: {len(bb_pop)} / {n}  wtd_mult: {wtd_mult:.6f}  ragu_gli: {ragu_gli:.4f}")

            lob_results[(lob, vintage)] = steps.to_dict()
            lob_flag_results[(lob, vintage)] = flags.to_dict()
            lob_wtd_mults[(lob, vintage)] = wtd_mult
            lob_record_counts[(lob, vintage)] = n

        output_df = build_output_df(lob_results, lob_wtd_mults, lob_record_counts, ragu_gli_dict)
        flags_df = build_flags_df(lob_flag_results, lob_record_counts)
        attribution_df = build_attribution_df(lob_results, ragu_gli_dict)
        diag_results_by_category[cat_name][lob] = (output_df, flags_df, attribution_df)

        if output_df is not None:
            print(f'\n--- {cat_name} / {lob} Multiplier Steps ---')
            display(output_df)
            print(f'\n--- {cat_name} / {lob} Flag Means ---')
            display(flags_df)
        if attribution_df is not None:
            print(f'\n--- {cat_name} / {lob} Gross Loss Attribution ---')
            display(attribution_df)


  DIAGNOSTICS: STE / STE

  STE / STE 2026 M03  (n=3774)
00_initial  | loss_multiplier mean: 1.000000
01_prev_co  | flag mean: 0.024377  | loss_multiplier mean: 1.002438
02_small_af | flag mean: 0.000000  | loss_multiplier mean: 1.002438
03_zero_cd  | flag mean: 0.074192  | loss_multiplier mean: 1.002438
04_high_mi  | flag mean: 0.012984  | loss_multiplier mean: 1.003739
05_high_pti | flag mean: 0.127186  | loss_multiplier mean: 1.003739
06_car_make | penalty: 0.004769  benefit: 0.037361  | loss_multiplier mean: 0.996728
07_theft    | flag mean: 0.000000  | loss_multiplier mean: 0.996728
08_mcy_low  | flag mean: 0.000000  | loss_multiplier mean: 0.996728
09_wknd/day | weekend: 0.264176  weekday: 0.735824  | loss_multiplier mean: 0.998165
10_stud_ln  | flag mean: 0.000000  | loss_multiplier mean: 0.980427
11_low_pti  | flag mean: 0.006889  | loss_multiplier mean: 0.979519
12_chime    | flag mean: 0.235559  | loss_multiplier mean: 1.009257
13_employ   | seasonal: 0.007684  waiter: 0.009

,STE | 2026 M03,STE | 2026 M04,STE | 2026 M05,STE | 2026 M06,STE | 2026 M07,STE | 2026 M08
00_initial,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
01_prev_aca_chargeoff,1.002438,1.001942,1.001762,1.001870,1.001557,1.001747
02_small_amt_financed,1.002438,1.001942,1.001762,1.001870,1.001557,1.001747
03_zero_cash_down,1.002438,1.001942,1.001762,1.001870,1.001557,1.001747
04_high_mileage,1.003739,1.003091,1.002333,1.002745,1.002366,1.002364
05_high_pti,1.003739,1.003091,1.002333,1.002745,1.002366,1.002364
06_car_make,0.996728,0.995796,0.994552,0.992623,0.994091,0.993289
07_theft_risk,0.996728,0.995796,0.994552,0.992623,0.994091,0.993289
08_mcy_low_mileage,0.996728,0.995796,0.994552,0.992623,0.994091,0.993289
09_weekend_weekday,0.998165,0.997684,0.997141,0.991327,0.994625,0.992018



--- STE / STE Flag Means ---


,STE | 2026 M03,STE | 2026 M04,STE | 2026 M05,STE | 2026 M06,STE | 2026 M07,STE | 2026 M08
prev_co_flag,0.024377,0.019424,0.017619,0.018704,0.015567,0.017472
small_amt_financed_flag,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
pricing_change_flag,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
zero_cash_down_flag,0.074192,0.089490,0.100476,0.130492,0.117684,0.127441
high_mileage_vehicle_flag,0.012984,0.011446,0.005714,0.008699,0.008095,0.006166
high_pti_flag,0.127186,0.101977,0.065714,0.075685,0.067248,0.071942
car_make_penalty_flag,0.004769,0.004162,0.003333,0.003480,0.003113,0.002055
car_make_benefit_flag,0.037361,0.038502,0.040476,0.052197,0.042964,0.046249
theft_risk_flag,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
mcy_low_mileage_flag,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



--- STE / STE Gross Loss Attribution ---


...

,STE | 2026 M03,STE | 2026 M04,STE | 2026 M05,STE | 2026 M06,STE | 2026 M07,STE | 2026 M08
00_initial,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
01_prev_aca_chargeoff,1.001964,1.000873,1.000923,1.000982,1.000601,1.001187
02_small_amt_financed,1.001964,1.000873,1.000923,1.000982,1.000601,1.001187
03_zero_cash_down,1.001964,1.000873,1.000923,1.000982,1.000601,1.001187
04_high_mileage,1.001964,1.000873,1.000923,1.000982,1.000601,1.001187
05_high_pti,1.001964,1.000873,1.000923,1.000982,1.000601,1.001187
06_car_make,0.989275,0.987260,0.988007,0.988805,0.990982,0.988131
07_theft_risk,0.989275,0.987260,0.988007,0.988805,0.990982,0.988131
08_mcy_low_mileage,0.989275,0.987260,0.988007,0.988805,0.990982,0.988131
09_weekend_weekday,0.989192,0.988595,0.990642,0.985796,0.989030,0.986166



--- Upcredit / STE Flag Means ---


,STE | 2026 M03,STE | 2026 M04,STE | 2026 M05,STE | 2026 M06,STE | 2026 M07,STE | 2026 M08
prev_co_flag,0.019637,0.008726,0.009225,0.009820,0.006012,0.011869
small_amt_financed_flag,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
pricing_change_flag,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
zero_cash_down_flag,0.051360,0.055846,0.064576,0.083470,0.092184,0.097923
high_mileage_vehicle_flag,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
high_pti_flag,0.064955,0.059337,0.027675,0.021277,0.022044,0.011869
car_make_penalty_flag,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
car_make_benefit_flag,0.063444,0.068063,0.064576,0.060556,0.048096,0.065282
theft_risk_flag,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
mcy_low_mileage_flag,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



--- Upcredit / STE Gross Loss Attribution ---


,STE | 2026 M03,STE | 2026 M04,STE | 2026 M05,STE | 2026 M06,STE | 2026 M07,STE | 2026 M08
Previous ACA Chargeoff,-0.042648,-0.020721,-0.021698,-0.027329,-0.027874,-0.023507
Small Amount Financed,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000
Zero Cash Down,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000
High Mileage Vehicle,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000
High PTI,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000
Car Make,0.277057,0.325323,0.305602,0.340796,0.448003,0.260126
Theft Risk,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000
MCY Low Mileage,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000
Weekend / Weekday,0.001826,-0.032104,-0.062666,0.084856,0.091438,0.039434
Student Loans,0.662145,0.723606,0.716736,0.848112,1.412617,0.603599


In [24]:
# =============================================================================
# CELL 9: EXCEL EXPORT
# =============================================================================

with pd.ExcelWriter(EXCEL_OUTPUT, engine='openpyxl') as writer:
    for cat_name in CATEGORIES:
        cat_diag = diag_results_by_category.get(cat_name, {})
        cat_results = results_by_category.get(cat_name, {})

        for lob in LOBS:
            diag_data = cat_diag.get(lob, (None, None, None))
            output_df, flags_df, attribution_df = diag_data

            sheet_mult = f'{cat_name} Mult Steps'[:31]
            sheet_flags = f'{cat_name} Flags'[:31]
            sheet_attr = f'{cat_name} Attribution'[:31]

            if output_df is not None:
                output_df.to_excel(writer, sheet_name=sheet_mult)
            if flags_df is not None:
                flags_df.to_excel(writer, sheet_name=sheet_flags)
            if attribution_df is not None:
                attribution_df.to_excel(writer, sheet_name=sheet_attr)

    summary_rows = []
    for cat_name in CATEGORIES:
        cat_results = results_by_category.get(cat_name, {})
        for lob in LOBS:
            model_df = cat_results.get(lob)
            if model_df is not None and len(model_df) > 0:
                lob_rows = model_df.reset_index()
                lob_rows = lob_rows[lob_rows.lob == lob]
                if len(lob_rows) > 0:
                    last_vintage = lob_rows.vintage.max()
                    last_row = lob_rows[lob_rows.vintage == last_vintage].iloc[0]
                    summary_rows.append({
                        'Category': cat_name,
                        'Latest Vintage': last_vintage,
                        'Contract MS': last_row.get('ms_original', float('nan')),
                        'Gross Loss Impact': last_row.get('gross_loss_impact', float('nan')),
                        'Recovery Impact': last_row.get('recovery_impact', float('nan')),
                        'LTV Impact': last_row.get('ltv_impact', float('nan')),
                        'APR Impact': last_row.get('apr_impact', float('nan')),
                        'RAGU Score': last_row.get('ragu_score', float('nan')),
                        'LTV': last_row.get('ltv', float('nan')),
                        'Loss Multiplier': last_row.get('loss_multiplier', float('nan')),
                    })
    if summary_rows:
        summary_df = pd.DataFrame(summary_rows).set_index('Category')
        summary_df.to_excel(writer, sheet_name='Summary')
        display(summary_df)

print(f"\nExported to {EXCEL_OUTPUT}")
print("[PROGRESS] Export Complete")


,Latest Vintage,Contract MS,Gross Loss Impact,Recovery Impact,LTV Impact,APR Impact,RAGU Score,LTV,Loss Multiplier
Category,,,,,,,,,
STE,2026 M08,137.664610,-0.124868,3.555638,6.071489,1.799471,148.966340,1.429470,1.004995
Downcredit,2026 M08,134.240358,-0.337233,3.316821,5.372858,0.311739,142.904544,1.474108,1.013489
Upcredit,2026 M08,143.907713,0.281744,4.024802,7.538639,4.511905,160.264803,1.344003,0.988730



Exported to STE_gl_diagnostics.xlsx
[PROGRESS] Export Complete
